## 0 · Use this checkout, not the installed package

Run this before anything else, and re-run it after a kernel restart.


In [1]:
# Setup — run this FIRST.
#
# Order matters. Jupyter imports whatever `shipit_agent` is installed in the
# kernel, which lags this checkout: `include_server_in_tool_names` exists in
# the repo and not in the released package, so RemoteMCPServer(...) raises
# TypeError on an argument that is genuinely there. Putting the repo ahead of
# site-packages BEFORE the first import is the whole fix — adjusting sys.path
# afterwards is too late, the wrong module is already cached.
import sys
import pathlib

repo = pathlib.Path.cwd().parent
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

import shipit_agent
print(f"shipit-agent {shipit_agent.__version__}")
print(f"from {shipit_agent.__file__}")
assert str(repo) in shipit_agent.__file__, (
    "still importing the installed copy — restart the kernel and run this cell first"
)

shipit-agent 1.6.0
from /Users/rahulraj/Documents/MYWORK/ai_developer/others/shipit_agent/shipit_agent/__init__.py


# Agent + MCP Token and Streaming Audit

End-to-end validation of the main `Agent` with Gemma 4 on Bedrock Mantle and the public DeepWiki MCP. This notebook prints the complete tool/event stream, measures token usage, verifies canonical output retention, and exercises failure and large-output paths.

## What this validates

- Streamable HTTP MCP discovery and calls
- Per-call MCP `_meta` without leaking it into tool arguments
- Every lifecycle event and every generated text chunk
- Canonical versus model-visible tool output sizes
- Prompt, completion, cache, and total token usage
- Bounded live chunks for very large tool output
- Structured `run_failed` events
- Safe skill activation without unrelated tool injection

In [2]:
import importlib.util
import json
import os
import sys
import time
from collections import Counter
from pathlib import Path

from shipit_agent import Agent, FunctionTool, format_event_line
from shipit_agent.llms import LLMResponse, LiteLLMChatLLM
from shipit_agent.mcp import MCPStreamableHTTPTransport, RemoteMCPServer
from shipit_agent.models import ToolCall
from shipit_agent.tools import ToolContext

print('Python:', sys.version.split()[0])
print('Workspace:', Path.cwd())

Python: 3.11.9
Workspace: /Users/rahulraj/Documents/MYWORK/ai_developer/others/shipit_agent/notebooks


## Load the supplied Gemma Mantle provider

The path and model are environment-overridable. No API key or credential is embedded in this notebook.

In [3]:
DEFAULT_PROVIDER = (
    '/Users/rahulraj/Documents/MYWORK/AFTDRK/CACHE/DRK_CACHE_BACK/'
    'drk_cache/llm/bedrock_mantle_provider.py'
)
PROVIDER = Path(os.getenv('SHIPIT_MANTLE_PROVIDER', DEFAULT_PROVIDER))
MODEL = os.getenv('SHIPIT_AUDIT_MODEL', 'bedrock-mantle/google.gemma-4-26b-a4b')
assert PROVIDER.exists(), f'Provider not found: {PROVIDER}'

spec = importlib.util.spec_from_file_location('bedrock_mantle_provider', PROVIDER)
module = importlib.util.module_from_spec(spec)
sys.modules['bedrock_mantle_provider'] = module
spec.loader.exec_module(module)
module.ensure_registered()

def new_llm():
    return LiteLLMChatLLM(model=MODEL)

print('Provider:', PROVIDER)
print('Model:', MODEL)

Provider: /Users/rahulraj/Documents/MYWORK/AFTDRK/CACHE/DRK_CACHE_BACK/drk_cache/llm/bedrock_mantle_provider.py
Model: bedrock-mantle/google.gemma-4-26b-a4b


## Connect DeepWiki MCP

DeepWiki is public and needs no authentication. The metadata resolver demonstrates generic trace context attached through MCP `_meta`.

In [4]:
DEEPWIKI_URL = 'https://mcp.deepwiki.com/mcp'

def new_deepwiki(*, namespaced=False):
    return RemoteMCPServer(
        name='deepwiki',
        transport=MCPStreamableHTTPTransport(DEEPWIKI_URL, timeout=120),
        include_server_in_tool_names=namespaced,
        tool_meta_resolver=lambda context, tool, arguments: {
            'trace_id': context.metadata.get('trace_id', 'not-set'),
            'client': 'shipit-notebook-audit',
        },
    )

server = new_deepwiki()
started = time.perf_counter()
try:
    discovered = server.discover_tools()
    print('Server:', server.server_info)
    print('Protocol:', server.protocol_version)
    print('Discovery seconds:', round(time.perf_counter() - started, 2))
    for tool in discovered:
        print('-', tool.name, json.dumps(tool.input_schema))
finally:
    server.close()

Server: {'name': 'DeepWiki', 'version': '2.14.3'}
Protocol: 2025-11-25
Discovery seconds: 1.2
- ask_question {"properties": {"repoName": {"anyOf": [{"type": "string"}, {"items": {"type": "string"}, "type": "array"}], "description": "GitHub repository or list of repositories (max 10) in owner/repo format."}, "question": {"description": "The question to ask about the repository.", "type": "string"}}, "required": ["repoName", "question"], "type": "object"}
- read_wiki_contents {"properties": {"repoName": {"description": "GitHub repository in owner/repo format (e.g. \"facebook/react\").", "type": "string"}}, "required": ["repoName"], "type": "object"}
- read_wiki_structure {"properties": {"repoName": {"description": "GitHub repository in owner/repo format (e.g. \"facebook/react\").", "type": "string"}}, "required": ["repoName"], "type": "object"}


## Direct MCP baseline

This isolates remote retrieval latency and response size from model overhead.

In [5]:
QUESTION = (
    'Concisely explain retry eligibility, backoff, and streaming cleanup. '
    'Name the core classes and methods and give two failure tests.'
)
server = new_deepwiki()
try:
    ask = next(tool for tool in server.discover_tools() if tool.name == 'ask_question')
    started = time.perf_counter()
    direct = ask.run(
        ToolContext(prompt=QUESTION, metadata={'trace_id': 'direct-baseline'}),
        repoName='openai/openai-python',
        question=QUESTION,
    )
    direct_seconds = time.perf_counter() - started
    print({
        'ok': direct.metadata.get('ok'),
        'seconds': round(direct_seconds, 2),
        'characters': len(direct.text),
        'words': len(direct.text.split()),
    })
    print(direct.text)
finally:
    server.close()

{'ok': True, 'seconds': 15.29, 'characters': 4596, 'words': 483}
The OpenAI Python SDK handles retry eligibility, exponential backoff, and streaming cleanup through the `BaseClient` class and its synchronous and asynchronous subclasses, `SyncAPIClient` and `AsyncAPIClient` respectively.   

## Retry Eligibility and Backoff

Retry eligibility is determined by the `_should_retry` method within the `BaseClient` class.  This method checks for specific HTTP status codes and a custom `x-should-retry` header. 

Requests are eligible for retry if:
*   The `x-should-retry` header is set to `"true"`. 
*   The HTTP status code is `408` (Request Timeout). 
*   The HTTP status code is `409` (Conflict, often indicating a lock timeout). 
*   The HTTP status code is `429` (Too Many Requests, i.e., rate limiting). 
*   The HTTP status code is `500` or greater (internal server errors). 

The `_calculate_retry_timeout` method, also in `BaseClient`, determines the delay before a retry attempt.  It impleme

## Build the main Agent

Only DeepWiki tools are exposed for this focused test. Complete results remain caller-visible; only the copy entering future model turns is subject to context budgets.

In [6]:

QUESTION = (
    'Concisely explain retry eligibility, backoff, and streaming cleanup. '
    'Name the core classes and methods and give two failure tests.'
)
def new_agent():
    return Agent(
        llm=new_llm(),
        mcps=[new_deepwiki(namespaced=True)],
        prompt=(
            'You are a precise repository research agent. Call DeepWiki before '
            'repository claims. Use declared arguments only. Ground the answer '
            'in canonical tool output.'
        ),
        name='gemma-deepwiki-notebook-audit',
        metadata={'trace_id': 'notebook-live-run'},
        max_iterations=6,
        progress_summaries=True,
        parallel_tool_execution=True,
        max_tool_concurrency=3,
        max_tool_output_chars=16_000,
        max_tool_output_group_chars=48_000,
        persist_large_tool_outputs=True,
        project_root='.',
    )

agent = new_agent()
TASK = (
    'Use DeepWiki ask_question on openai/openai-python. ' + QUESTION +
    ' Call exactly one MCP tool unless it fails.'
)
selected = agent._selected_skills(TASK)
print('Selected skills:', [skill.id for skill in selected])
print('Effective max iterations:', agent._effective_max_iterations(selected))

Selected skills: []
Effective max iterations: 6


## Complete raw event stream

The MCP body is printed once from `tool_output_delta`. Final text chunks print continuously as they arrive. Heavy canonical MCP metadata is intentionally not duplicated into every delta.

In [7]:
TASK = (
    'Use DeepWiki ask_question on openai/openai-python. ' + QUESTION +
    ' Call exactly one MCP tool unless it fails.'
)
narrated =  Agent(
        llm=new_llm(),
        mcps=[new_deepwiki(namespaced=True)],
        prompt=(
            'You are a precise repository research agent. Call DeepWiki before '
            'repository claims. Use declared arguments only. Ground the answer '
            'in canonical tool output.'
        ),
        name='gemma-deepwiki-notebook-audit',
        metadata={'trace_id': 'notebook-live-run'},
        max_iterations=6,
        progress_summaries=True,
        parallel_tool_execution=True,
        max_tool_concurrency=3,
        max_tool_output_chars=16_000,
        max_tool_output_group_chars=48_000,
        persist_large_tool_outputs=True,
        project_root='.',
)
narrated_events = []
for event in narrated.stream(TASK):
    narrated_events.append(event)
    if event.type == "agent_decision":
        print(f"\n▸ {event.payload['summary']}")
    elif event.type == "agent_observation":
        print(f"  ← {event.payload['summary']}")
    elif event.type == "progress_summary_failed":
        print(f"  ! {event.payload.get('error', '')[:140]}")


▸ Asking question openai/openai-python.
  ← Asked question openai/openai-python.


watcher: otlp export failed 500: b'{"message":"Internal Server Error","error":"An unknown error occurred"}'


In [8]:
for event in narrated_events:
    print("event", repr(event))

event AgentEvent(type='run_started', message='Agent run started', payload={'prompt': 'Use DeepWiki ask_question on openai/openai-python. Concisely explain retry eligibility, backoff, and streaming cleanup. Name the core classes and methods and give two failure tests. Call exactly one MCP tool unless it fails.'}, timestamp=1786360099.715005)
event AgentEvent(type='mcp_attached', message='MCP server attached: deepwiki', payload={'server': 'deepwiki'}, timestamp=1786360099.718257)
event AgentEvent(type='step_started', message='LLM completion started', payload={'tool_count': 3, 'iteration': 1}, timestamp=1786360099.7183738)
event AgentEvent(type='usage_tick', message='Usage updated', payload={'usage': {'prompt_tokens': 574, 'completion_tokens': 77, 'total_tokens': 651, 'cache_read_input_tokens': 0, 'cache_creation_input_tokens': 0}, 'iteration': 1}, timestamp=1786360108.549262)
event AgentEvent(type='agent_decision', message='Asking question openai/openai-python.', payload={'summary': 'Ask

In [ ]:
event_counts = Counter()
completed_payload = {}
tool_telemetry = []
events = []
in_text = False
started = time.perf_counter()

for event in agent.stream(TASK):
    # print('event', event)
    events.append(event)
    event_counts[event.type] += 1
    payload = event.payload
    if event.type == 'text_delta':
        if not in_text:
            print('\ntext_delta stream:')
            in_text = True
        print(payload.get('chunk', ''), end='', flush=True)
        continue
    if in_text:
        print('\n[end text_delta stream]')
        in_text = False
    if event.type == 'tool_output_delta':
        chunk = str(payload.get('chunk', ''))
        print('tool_output_delta', {
            'tool': payload.get('tool'),
            'sequence': payload.get('sequence'),
            'characters': len(chunk),
            'metadata': payload.get('chunk_metadata'),
        })
        print(chunk)
    elif event.type == 'tool_completed':
        telemetry = {
            'tool': payload.get('tool'),
            'output_chars': payload.get('output_chars'),
            'model_output_chars': payload.get('model_output_chars'),
            'model_output_reduced': payload.get('model_output_reduced'),
            'metadata': payload.get('metadata'),
        }
        tool_telemetry.append(telemetry)
        print('tool_completed', json.dumps(telemetry, default=str))
    elif event.type == 'run_completed':
        completed_payload = dict(payload)
        print('run_completed', {
            'usage': payload.get('usage'),
            'output_chars': len(str(payload.get('output', ''))),
            'cancelled': payload.get('cancelled'),
        })
    else:
        display_line = format_event_line(event)
        if display_line:
            print(display_line)
        else:
            safe = {
                key: value for key, value in payload.items()
                if key not in {'output', 'content', 'chunk'}
            }
            print(event.type, json.dumps(safe, ensure_ascii=False, default=str))

if in_text:
    print('\n[end text_delta stream]')
agent_seconds = time.perf_counter() - started

event run_started          Agent run started
event mcp_attached         MCP server attached: deepwiki
event step_started         LLM completion started
event usage_tick           Usage updated
event agent_decision       Asking question openai/openai-python.
event tool_group_started   Tool group started
event tool_called          Tool called: deepwiki__ask_question
event tool_output_started  Tool output started: deepwiki__ask_question
event tool_output_delta    Tool output: deepwiki__ask_question
event tool_completed       Tool completed: deepwiki__ask_question
event tool_group_completed Asked question openai/openai-python.
event agent_observation    Asked question openai/openai-python.
event step_started         LLM completion started
event text_delta           In
event text_delta
event text_delta           `openai
event text_delta           -
event text_delta           python
event text_delta           `,
event text_delta            the
event text_delta            retry
event text_del

watcher: otlp export failed 500: b'{"message":"Internal Server Error","error":"An unknown error occurred"}'


In [9]:
# Full old-style diagnostic output: payloads, tool results, telemetry, timestamps.
# This reprints the captured run; it does not call the model or MCP again.
for event in events:
    print('event', repr(event))

[AgentEvent(type='run_started', message='Agent run started'),
 AgentEvent(type='mcp_attached', message='MCP server attached: deepwiki'),
 AgentEvent(type='step_started', message='LLM completion started'),
 AgentEvent(type='usage_tick', message='Usage updated'),
 AgentEvent(type='agent_decision', message='Asking question openai/openai-python.'),
 AgentEvent(type='tool_group_started', message='Tool group started'),
 AgentEvent(type='tool_called', message='Tool called: deepwiki__ask_question'),
 AgentEvent(type='tool_output_started', message='Tool output started: deepwiki__ask_question'),
 AgentEvent(type='tool_output_delta', message='Tool output: deepwiki__ask_question'),
 AgentEvent(type='tool_completed', message='Tool completed: deepwiki__ask_question'),
 AgentEvent(type='tool_group_completed', message='Asked question openai/openai-python.'),
 AgentEvent(type='agent_observation', message='Asked question openai/openai-python.'),
 AgentEvent(type='step_started', message='LLM completion s

## Measured run report

In [8]:
# Human-facing progress only. The complete low-level trace remains in `events`.
progress_types = {
    'skills_selected', 'agent_decision', 'tool_called', 'agent_observation',
    'tool_failed', 'run_failed', 'run_completed',
}
def progress_line(event):
    return (
        format_event_line(event)
        or f'{event.type:<20} {event.display_message}'.rstrip()
    )

progress_transcript = [
    progress_line(event) for event in events if event.type in progress_types
]
print('\n'.join(progress_transcript))
print('\nRaw event counts:', dict(event_counts))

agent_decision       Asking question openai/openai-python.
tool_called          Tool called: deepwiki__ask_question
agent_observation    Asked question openai/openai-python.
run_completed        Agent run completed

Raw event counts: {'run_started': 1, 'mcp_attached': 1, 'step_started': 2, 'usage_tick': 2, 'agent_decision': 1, 'tool_group_started': 1, 'tool_called': 1, 'tool_output_started': 1, 'tool_output_delta': 1, 'tool_completed': 1, 'tool_group_completed': 1, 'agent_observation': 1, 'text_delta': 393, 'final_answer': 1, 'run_completed': 1}


In [ ]:
usage = completed_payload.get('usage', {})
token_report = {
    'prompt_tokens': usage.get('prompt_tokens', 0),
    'completion_tokens': usage.get('completion_tokens', 0),
    'total_tokens': usage.get('total_tokens', 0),
    'cache_read_input_tokens': usage.get('cache_read_input_tokens', 0),
    'cache_creation_input_tokens': usage.get('cache_creation_input_tokens', 0),
}
run_report = {
    'wall_seconds': round(agent_seconds, 2),
    'event_counts': dict(event_counts),
    'usage': token_report,
    'final_output_chars': len(str(completed_payload.get('output', ''))),
    'tool_telemetry': tool_telemetry,
}
print(json.dumps(run_report, indent=2, default=str))
print('\nFINAL ANSWER\n')
print(completed_payload.get('output', ''))

## Large-output streaming stress test

This local deterministic test proves that a 100,000-character result stays complete while live deltas remain bounded and the model-visible copy is capped.

In [ ]:
class OneToolThenAnswer:
    def __init__(self):
        self.calls = 0

    def complete(self, **kwargs):
        self.calls += 1
        if self.calls == 1:
            return LLMResponse(
                content='',
                tool_calls=[ToolCall(name='large_result', arguments={})],
                usage={'prompt_tokens': 10, 'completion_tokens': 2, 'total_tokens': 12},
            )
        return LLMResponse(
            content='Large result processed.',
            usage={'prompt_tokens': 20, 'completion_tokens': 4, 'total_tokens': 24},
        )

large_text = '0123456789' * 10_000
large_agent = Agent(
    llm=OneToolThenAnswer(),
    tools=[FunctionTool.from_callable(lambda: large_text, name='large_result')],
    auto_use_skills=False,
    max_tool_output_chars=4_000,
    max_tool_output_group_chars=4_000,
)
large_events = list(large_agent.stream('Run the large result tool.'))
large_deltas = [e.payload['chunk'] for e in large_events if e.type == 'tool_output_delta']
large_completed = next(e for e in large_events if e.type == 'tool_completed')
print({
    'canonical_chars': large_completed.payload['output_chars'],
    'model_chars': large_completed.payload['model_output_chars'],
    'model_output_reduced': large_completed.payload['model_output_reduced'],
    'delta_count': len(large_deltas),
    'largest_delta': max(map(len, large_deltas)),
    'reassembled_complete': ''.join(large_deltas) == large_text,
})

## Structured failure stream

Provider failures still raise to the caller, but consumers receive a terminal `run_failed` event first.

In [ ]:
class BrokenLLM:
    def complete(self, **kwargs):
        raise ConnectionError('simulated provider outage')

failure_events = []
stream = Agent(llm=BrokenLLM(), auto_use_skills=False).stream('test failure')
try:
    while True:
        event = next(stream)
        failure_events.append(event)
        print(event.type, event.payload)
except StopIteration:
    pass
except ConnectionError as exc:
    print('Expected exception:', exc)

failed = [event for event in failure_events if event.type == 'run_failed']
assert len(failed) == 1
assert failed[0].payload['retryable'] is True

## Skill activation audit

Fuzzy catalog search is explicit. Runtime auto-activation only uses authored trigger phrases, preventing unrelated prompts and tool bundles from inflating every model turn.

In [ ]:
skill_agent = Agent(llm=OneToolThenAnswer())
mcp_prompt = 'Use DeepWiki MCP to analyze retry behavior in a repository.'
trigger_prompt = 'Please debug this production bug and plan this feature.'
print('MCP prompt skills:', [s.id for s in skill_agent._selected_skills(mcp_prompt)])
print('Authored-trigger skills:', [s.id for s in skill_agent._selected_skills(trigger_prompt)])
print('Explicit fuzzy search:', [s.id for s in skill_agent.search_skills('database')[:5]])
assert skill_agent._selected_skills(mcp_prompt) == []
assert skill_agent._selected_skills(trigger_prompt)

## Final assertions

In [ ]:
checks = {
    'deepwiki_direct_ok': direct.metadata.get('ok') is True,
    'agent_completed': bool(completed_payload.get('output')),
    'one_mcp_call': event_counts['tool_called'] == 1,
    'usage_reported': usage.get('total_tokens', 0) > 0,
    'canonical_result_complete': all(t['output_chars'] > 0 for t in tool_telemetry),
    'large_result_complete': ''.join(large_deltas) == large_text,
    'large_model_copy_reduced': large_completed.payload['model_output_reduced'],
    'failure_event_emitted': len(failed) == 1,
    'no_unrelated_skill_injection': skill_agent._selected_skills(mcp_prompt) == [],
}
for name, passed in checks.items():
    print(f'{"PASS" if passed else "FAIL"}: {name}')
assert all(checks.values()), checks
print('\nAll agent, MCP, streaming, token, and failure checks passed.')